# 02 — Data Cleaning Pipeline

In this notebook, we're going to clean up the NYC Airbnb data based on our findings from the extraction phase. 

**Our Cleaning Checklist:**
1. **Fix column names**: Make them easier to work with (snake_case).
2. **Handle missing values**: Fill in blanks for reviews and names.
3. **Fix data types**: Ensure dates are actually dates.
4. **Handle outliers**: Remove $0 prices and cap extreme values.
5. **Feature engineering**: Add some useful columns like `price_tier` and `revenue_proxy` for our dashboard.

### Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path so we can import our cleaning script
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from scripts.etl_pipeline import (
    normalize_columns, rename_verbose_columns, drop_duplicates,
    convert_types, fill_missing, treat_price_outliers, 
    treat_min_nights, engineer_features, FINAL_COLUMNS
)

raw_path = project_root / 'data' / 'raw' / 'AB_NYC_2019.csv'
processed_path = project_root / 'data' / 'processed' / 'airbnb_nyc_cleaned.csv'

### Loading and Standardizing
First, we'll load the data and fix those long column names.

In [2]:
df = pd.read_csv(raw_path)
print(f"Initial shape: {df.shape}")

df = normalize_columns(df)
df = rename_verbose_columns(df)
df.head(2)

Initial shape: (48895, 16)
2026-05-04 15:04:21,025 [INFO] Step: Normalizing column names to snake_case.
2026-05-04 15:04:21,029 [INFO] Step: Applying verbose column renames for KPI consistency.


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,review_count,last_review,review_rate_month,host_listing_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355


### Cleaning Phase
Now we'll run through our cleaning steps: removing duplicates, fixing types, and handling missing data.

In [3]:
df = drop_duplicates(df)
df = convert_types(df)
df = fill_missing(df)

print(f"Shape after basic cleaning: {df.shape}")

2026-05-04 15:04:23,469 [INFO] Step: Handling duplicates.
2026-05-04 15:04:23,529 [INFO] Step: Converting data types (category/datetime/string).
2026-05-04 15:04:23,551 [INFO] Step: Imputing missing values.
2026-05-04 15:04:23,555 [INFO] Handled nulls in 'name', 'host_name', and 'review_rate_month'. 'last_review' kept as NaT.
Shape after basic cleaning: (48895, 16)


### Fixing Outliers
We'll remove the $0 listings and cap the price at the 99th percentile to avoid skewing our analysis.

In [4]:
df = treat_price_outliers(df)
df = treat_min_nights(df)

print(f"Final price range: ${df['price'].min()} to ${df['price'].max()}")

2026-05-04 15:04:25,066 [INFO] Step: Treating price outliers.
2026-05-04 15:04:25,083 [WARNING] Removed 11 records with $0 price.
2026-05-04 15:04:25,089 [INFO] Price capped at 99th percentile: $799.
2026-05-04 15:04:25,090 [INFO] Step: Normalizing minimum stay requirements.
Final price range: $10 to $799


### Feature Engineering
Let's add some extra columns that will make our analysis easier later on.

In [5]:
df = engineer_features(df)
df[['price', 'price_tier', 'revenue_proxy', 'occupancy_rate_est']].head()

2026-05-04 15:04:26,826 [INFO] Step: Engineering analytical features and KPIs.


,price,price_tier,revenue_proxy,occupancy_rate_est
0,47,Budget,15651,0.0877
1,49,Budget,14357,0.1973
2,49,Budget,16758,0.0630
3,49,Budget,16611,0.0712
4,47,Budget,8131,0.5260


### Save the Results
Finally, we'll select our core columns and save the cleaned dataset.

In [6]:
df_final = df[[c for c in FINAL_COLUMNS if c in df.columns]].copy()
df_final.to_csv(processed_path, index=False)

print(f"Success! Cleaned data saved to {processed_path}")
print(f"Final listing count: {len(df_final):,}")

Success! Cleaned data saved to /Users/meghna/Desktop/E_G18_NYCAirbnbAnalysis/data/processed/airbnb_nyc_cleaned.csv
Final listing count: 48,884
